In [11]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

X = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv', index_col='id') 
X_test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv', index_col='id')

X.dropna(axis=0, subset=['Will_Buy_EV'], inplace=True)

y = X['Will_Buy_EV'].map({'Yes': 1, 'No': 0})

X.drop(['Will_Buy_EV'], axis=1, inplace=True)

X.head()

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
id,,,,,,,,,,,,,
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low


In [15]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Identify categorical and numerical columns directly from X
categorical_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
numerical_cols = X.select_dtypes(exclude=['object', 'bool']).columns.tolist()

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [22]:
from sklearn.model_selection import train_test_split, GridSearchCV

# Sample 10% of the data for fast hyperparameter tuning
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.1, random_state=0)

def create_model(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=0,
        n_jobs=-1
    )

    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
# Define your parameter grid (with 'model__' prefix for pipeline steps)
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

# Instantiate the pipeline from your create_model function
pipeline = create_model()

# Setup Grid Search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Run the search
grid_search.fit(X_sample, y_sample)

print("Best CV Accuracy:", grid_search.best_score_)
print("Best Parameters:", grid_search.best_params_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


KeyboardInterrupt: 

Results without XGBoost: 

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best CV Accuracy: 0.8949690519096359
Best Parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}

## XGBoost

In [21]:
from xgboost import XGBClassifier

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

def create_model(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    min_child_weight=1
):
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        random_state=0,
        n_jobs=2,  # Stabiler in Notebooks, um Deadlocks zu vermeiden
        eval_metric='logloss'
    )

    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

param_grid = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [3, 6, 10],
    'model__learning_rate': [0.05, 0.1]
}

pipeline = create_model()

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print("Best CV Accuracy:", grid_search.best_score_)
print("Best Parameters:", grid_search.best_params_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best CV Accuracy: 0.8983167960897062
Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': 6, 'model__n_estimators': 300}


Results with XGBoost:
```
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best CV Accuracy: 0.8983167960897062
Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': 6, 'model__n_estimators': 300}
```